# 📊 BÀI TOÁN TÍCH HỢP DỮ LIỆU GIAO THÔNG, THỜI TIẾT VÀ LỊCH LỄ
**Môn học**: IT4930 - Khoa học dữ liệu (Data Science)  
**Mục tiêu**: Kết hợp dữ liệu lưu lượng giao thông (`traffic_volume`) với dữ liệu thời tiết (`weather`) và lịch ngày lễ (`holidays`) thành một bảng dữ liệu hợp nhất chuẩn mực theo mốc thời gian chung.

---

### 📌 Các bước thực hiện chính:
1. **Xác định khóa chung**: Sử dụng trường thời gian (`date_time` / `timestamp`) làm khóa liên kết duy nhất giữa các bảng.
2. **Chuẩn hóa thời gian (ISO-8601 & Timezone)**:
   - Parse các định dạng ngày giờ hỗn hợp (`M/D/YY H:MM`, `YYYY-MM-DD HH:MM:SS`) về chuẩn quốc tế **ISO-8601**.
   - Đồng bộ hóa múi giờ (`America/Chicago` - US Central Time) và xử lý biến động chuyển đổi giờ mùa hè (**Daylight Saving Time - DST**).
   - Tạo biểu diễn mốc thời gian chuẩn quốc tế **UTC ISO-8601** (`YYYY-MM-DDTHH:MM:SSZ`).
3. **Xử lý quan hệ 1-N & Chống nhân bản**: Xử lý hiện tượng một mốc giờ có nhiều bản ghi thời tiết bằng kỹ thuật gộp thông minh (aggregation), tránh nhân đôi bản ghi giao thông sai lệch.
4. **Phát tán thông tin ngày lễ**: Broadcast nhãn ngày lễ cho toàn bộ 24 giờ trong ngày và gán nhãn phân loại `day_type` (`Weekday`, `Weekend`, `Holiday`).
5. **Thực hiện ghép nối (Join)**: Áp dụng phép toán **Left Join** (hoặc **Inner Join**) với bảng dữ liệu giao thông làm bảng chủ đạo.
6. **Kiểm định chất lượng dữ liệu (QA)**: Đảm bảo tính duy nhất của khóa, không lệch mốc thời gian, không phát sinh trùng lặp bản ghi.
7. **Xuất file trung gian**: Lưu vào `data/interim/merged_traffic_data.csv`.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import pytz
import matplotlib.pyplot as plt

# Thêm đường dẫn src vào hệ thống để import module tích hợp dữ liệu
sys.path.append(os.path.abspath('..'))
from src.data_merge import (
    load_dataset,
    extract_raw_tables,
    standardize_timestamps,
    aggregate_weather_data,
    enrich_temporal_features,
    merge_traffic_weather_holidays,
    validate_integration_quality,
    export_data
)

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
print("Import thành công! Pandas version:", pd.__version__)


## 1. Nạp và Khám phá Dữ liệu Nguồn (Raw Data Exploration)
Tập dữ liệu ghi nhận lưu lượng xe trên tuyến đường cao tốc I-94 (Minneapolis - St. Paul, Minnesota) từ năm 2012 đến 2018 kèm theo các điều kiện khí tượng và ngày nghỉ lễ.


In [ ]:
RAW_FILE = '../dataset/Metro_Interstate_Traffic_Volume_modified.csv'
df_raw = pd.read_csv(RAW_FILE)

print(f"Kích thước tập dữ liệu thô: {df_raw.shape[0]:,} dòng, {df_raw.shape[1]} cột")
print("\nThông tin cấu trúc các cột dữ liệu:")
df_raw.info()
df_raw.head()


### 🔍 Phân tích các vấn đề kỹ thuật trong dữ liệu thô:
1. **Định dạng thời gian không chuẩn**: Dữ liệu thời gian ở định dạng chuỗi `10/2/12 9:00` (Month/Day/Year Hour:Minute), chưa theo chuẩn ISO-8601 và chưa có timezone offset.
2. **Trùng lặp mốc thời gian do đa hiện tượng thời tiết (1-to-Many relationship)**:
   - Toàn bộ dataset có 48,204 dòng nhưng chỉ có 40,575 mốc thời gian duy nhất.
   - Có 7,629 dòng trùng lặp mốc giờ do trạm thời tiết ghi nhận nhiều hiện tượng đồng thời (ví dụ: vừa có sương mù `Mist` vừa có `Fog` hoặc `Snow` và `Haze`).
3. **Dữ liệu ngày lễ chỉ gán ở giờ đầu tiên**:
   - Cột `holiday` chỉ có 61 giá trị không rỗng (chỉ xuất hiện lúc `00:00:00`), 23 giờ còn lại trong ngày lễ đều bị để trống (`NaN`).


In [ ]:
print("Tổng số dòng:", len(df_raw))
print("Số mốc thời gian duy nhất:", df_raw['date_time'].nunique())
print("Số dòng trùng lặp mốc thời gian:", df_raw['date_time'].duplicated().sum())

# Minh họa trường hợp trùng lặp mốc thời gian do nhiều hiện tượng thời tiết
sample_dup = df_raw[df_raw['date_time'] == '1/10/14 20:00']
print("\nVí dụ mốc thời gian '1/10/14 20:00' có 2 bản ghi thời tiết (Mist và Fog):")
display(sample_dup[['date_time', 'weather_main', 'weather_description', 'temp', 'fog_mm', 'wind_speed_ms', 'traffic_volume']])


## 2. Phân tách Dữ liệu thành các Bảng Nguồn Riêng Biệt (Domain Separation)
Trong hệ thống thực tế, dữ liệu đến từ 3 nguồn độc lập:
- **Hệ thống cảm biến lưu lượng xe (Traffic Sensor System)**: Bảng `traffic_data_raw.csv`.
- **Trạm quan trắc khí tượng (Meteorological Station)**: Bảng `weather_data_raw.csv`.
- **Lịch biểu hành chính & Ngày lễ (Holiday Calendar System)**: Bảng `holiday_calendar_raw.csv`.


In [ ]:
traffic_raw, weather_raw, holiday_raw = extract_raw_tables(df_raw, raw_dir='../data/raw')

print(f"- Bảng giao thông thô (Traffic): {traffic_raw.shape[0]:,} dòng")
print(f"- Bảng thời tiết thô (Weather):   {weather_raw.shape[0]:,} dòng")
print(f"- Bảng ngày lễ thô (Holidays):    {holiday_raw.shape[0]:,} ngày lễ")


## 3. Chuẩn hóa Thời gian (ISO-8601) & Xử lý Múi giờ (Timezone & DST)
Dữ liệu được thu thập tại Minnesota thuộc múi giờ **US Central Time (`America/Chicago`)**:
- Giờ chuẩn mùa đông (CST): `UTC-6`
- Giờ mùa hè (CDT - Daylight Saving Time): `UTC-5`

Chúng ta chuẩn hóa các mốc thời gian về các định dạng chuẩn:
1. `date_time_iso`: `YYYY-MM-DD HH:MM:SS` (Dễ đọc, chuẩn hóa độ dài cố định, thuận tiện truy vấn).
2. `date_time_local`: `YYYY-MM-DDTHH:MM:SS-05:00` / `-06:00` (ISO-8601 có timezone offset rõ ràng).
3. `date_time_utc`: `YYYY-MM-DDTHH:MM:SSZ` (ISO-8601 quốc tế chuẩn hóa toàn cầu).
4. `date`: `YYYY-MM-DD` (Khóa ngày dùng để map với lịch nghỉ lễ).


In [ ]:
traffic_std = standardize_timestamps(traffic_raw, 'date_time', source_tz='America/Chicago')
weather_std = standardize_timestamps(weather_raw, 'date_time', source_tz='America/Chicago')

print("Mẫu dữ liệu giao thông sau chuẩn hóa thời gian:")
display(traffic_std[['date_time_raw', 'date_time_iso', 'date_time_local', 'date_time_utc', 'date', 'traffic_volume']].head(5))


## 4. Xử lý Quan hệ 1-N trong Dữ liệu Thời tiết (Weather Aggregation)
Nếu thực hiện phép nối trực tiếp khi 1 mốc giờ có nhiều bản ghi thời tiết, bảng giao thông sẽ bị nhân đôi số dòng (Cartesian product / duplicate rows), làm biến dạng các phép tính tổng và trung bình lưu lượng xe.

**Quy tắc gộp (Aggregation Rules)**:
- **Biến số** (`temp`, `rain_1h`, `snow_1h`, `clouds_all`, `fog_mm`, `wind_speed_ms`, `flood_mm`): Tính trung bình (`mean`).
- **Biến phân loại** (`weather_main`, `weather_description`): Kết hợp các nhãn duy nhất bằng dấu phân cách `/` và `; `.


In [ ]:
weather_agg = aggregate_weather_data(weather_std, time_key='date_time_iso')

print(f"Kích thước bảng thời tiết sau khi gộp: {weather_agg.shape[0]:,} dòng (1 dòng duy nhất cho mỗi mốc giờ)")
# Kiểm tra mốc giờ '2014-01-10 20:00:00' đã được gộp gọn gàng
sample_agg = weather_agg[weather_agg['date_time_iso'] == '2014-01-10 20:00:00']
display(sample_agg)


## 5. Xử lý Lịch Ngày Lễ (Full-Day Holiday Mapping & Day Type)
Một ngày lễ có hiệu lực trọn vẹn 24 giờ. Bằng cách nối theo khóa ngày (`date` - `YYYY-MM-DD`), toàn bộ các giờ trong ngày lễ đều nhận đúng tên ngày lễ và cờ `is_holiday = 1`.

Đồng thời, chúng ta phân loại biến `day_type`:
- `Holiday`: Ngày lễ chính thức.
- `Weekend`: Thứ 7, Chủ nhật (không trùng ngày lễ).
- `Weekday`: Thứ 2 đến Thứ 6 (không trùng ngày lễ).


In [ ]:
print("Danh sách các ngày lễ tiêu biểu trong lịch:")
display(holiday_raw.head(10))


## 6. Thực hiện Ghép nối Dữ liệu (Join Operations)
Áp dụng phép toán **Left Join** tuần tự:
$$\text{Traffic\_Unique} \underset{\text{date\_time\_iso}}{\bowtie} \text{Weather\_Agg} \underset{\text{date}}{\bowtie} \text{Holiday\_Calendar}$$


In [ ]:
merged_df = merge_traffic_weather_holidays(
    traffic_df=traffic_raw,
    weather_df=weather_raw,
    holiday_df=holiday_raw,
    join_type='left',
    aggregate_weather=True
)

print(f"Kích thước bảng hợp nhất: {merged_df.shape[0]:,} dòng, {merged_df.shape[1]} cột.")
display(merged_df.head(5))


## 7. Kiểm định Chất lượng Dữ liệu (Quality Assurance & Validation)
Kiểm tra các tiêu chí chất lượng nghiêm ngặt:
- Khóa thời gian không bị trùng lặp (Duplicate timestamps = 0).
- Không bị mất mát dữ liệu chuỗi thời gian giao thông.
- Phân phối loại ngày và ngày lễ hợp lý.


In [ ]:
qa_metrics = validate_integration_quality(merged_df, traffic_raw, weather_raw, holiday_raw)

print("\n--- BÁO CÁO KIỂM ĐỊNH TỔNG HỢP ---")
for k, v in qa_metrics.items():
    print(f"{k}: {v}")


## 8. Phân tích Trực quan Dữ liệu sau Tích hợp (Exploratory Data Visualization)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Lưu lượng giao thông trung bình theo giờ trong ngày cho từng loại ngày
hourly_trend = merged_df.groupby(['hour', 'day_type'])['traffic_volume'].mean().unstack()
hourly_trend.plot(ax=axes[0], marker='o', linewidth=2.5)
axes[0].set_title('Lưu lượng giao thông trung bình theo giờ trong ngày', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Giờ trong ngày (0 - 23)', fontsize=10)
axes[0].set_ylabel('Lưu lượng trung bình (xe/giờ)', fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. Lưu lượng giao thông theo nhóm thời tiết
weather_trend = merged_df.groupby('weather_main')['traffic_volume'].mean().sort_values(ascending=False).head(8)
weather_trend.plot(kind='bar', ax=axes[1], color='teal', edgecolor='black', alpha=0.8)
axes[1].set_title('Lưu lượng giao thông trung bình theo điều kiện thời tiết', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nhóm thời tiết', fontsize=10)
axes[1].set_ylabel('Lưu lượng trung bình (xe/giờ)', fontsize=10)
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


## 9. Xuất File Dữ liệu Trung gian (Interim Dataset Export)


In [ ]:
OUTPUT_PATH = '../data/interim/merged_traffic_data.csv'
export_data(merged_df, OUTPUT_PATH)

print(f"File trung gian đã được lưu hoàn tất tại: {OUTPUT_PATH}")
